# Preisanalyse von Gebrauchtwagen in Deutschland (AutoScout24, 2023)

## 1. Einleitung

Dieser Datensatz stammt von Kaggle und ist die Grundlage dieser Analyse.
Der Datensatz "Germany Used Cars Dataset 2023" ist von der Plattform Autoscout24 und hat über 200.000 Inserate (Beobachtungen) vom Zeitraum 1995 bis 2023.


(Quelle: wspirat, 2023, Kaggle, CC0)
https://www.kaggle.com/datasets/wspirat/germany-used-cars-dataset-2023/data

Ziel der Analyse ist es herauszufinden, welche Merkmale den Preis eines Gebrauchtwagens
bestimmen und wie gut sich der Preis vorhersagen lässt.
Diese Frage ist für einen Gebrauchtwagenhändler relevant, da er seine Fahrzeuge
marktgerecht bepreisen und günstige Einkaufsmöglichkeiten erkennen möchte.

Daraus ergeben sich zwei Forschungsfragen:

1. Welche Merkmale beeinflussen den Preis am stärksten?
2. Wie gut lässt sich der Preis anhand der Merkmale vorhersagen?

Als Zielvariable dient der Angebotspreis (`price_in_euro`).

## 2. Setup

In [ ]:
# Imports
from pickle import TRUE

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Darstellung
%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# relative Pfade (reproduzierbar)
DATA_RAW = Path("../data/raw/data.csv")
DATA_PROCESSED = Path("../data/processed/data_processed.csv")
IMG_DIR = Path("../images/eda")
IMG_DIR.mkdir(parents=True ,exist_ok=True)

### Hilfsfunktionenen

In [ ]:
# Preise in Euro-Format
def euro(x, decimals=2):
    return f"{x:,.{decimals}f} €".replace(",", "X").replace(".", ",").replace("X", ".")


# Grafiken werden einheitlich und sortiert abgespeichert
def plot_verteilung(daten, titel, xlabel, fname, bins=50, binrange=None, discrete=False, xlim=None, xticks=None):
    plt.figure(figsize=(9, 4.5))
    sns.histplot(daten, bins=bins, binrange=binrange, discrete=discrete, color="steelblue")
    plt.title(titel); plt.xlabel(xlabel); plt.ylabel("Anzahl")
    if xlim:
        plt.xlim(*xlim)
    if xticks is not None: 
        plt.xticks(xticks)
    plt.tight_layout()
    plt.savefig(IMG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


# Häufigkeiten kategorialer Merkmale als horizontale Balken
def plot_kategorie(spalte, titel, fname, top=None, capitalize=False, mapping=None):
    counts = df_clean[spalte].value_counts()
    if top:
        counts = counts.head(top)
    if mapping:
        counts.index = [mapping.get(x, x) for x in counts.index]
    elif capitalize:
        counts.index = counts.index.str.capitalize()
    plt.figure(figsize=(9, max(3, len(counts) * 0.35)))
    ax = sns.barplot(x=counts.values, y=counts.index, color="steelblue")
    ax.bar_label(ax.containers[0],
                 labels=[f"{int(v):,}".replace(",", ".") for v in counts.values],
                 padding=3, fontsize=9)
    ax.grid(False)
    ax.set_xticks([])
    plt.margins(x=0.12)
    plt.title(titel); plt.xlabel("Anzahl"); plt.ylabel("")
    plt.tight_layout()
    plt.savefig(IMG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()


# Medianpreis je Kategorie als horizontale Balken
def plot_preis_je_kategorie(spalte, titel, fname, top_haeufig=None, capitalize=False, mapping=None):
    daten = df_clean
    if top_haeufig:
        haeufig = daten[spalte].value_counts().head(top_haeufig).index
        daten = daten[daten[spalte].isin(haeufig)]
    med = daten.groupby(spalte)["price_in_euro"].median().sort_values(ascending=False)
    if mapping:
        med.index = [mapping.get(x, x) for x in med.index]
    elif capitalize:
        med.index = med.index.str.capitalize()
    plt.figure(figsize=(9, max(3, len(med) * 0.35)))
    ax = sns.barplot(x=med.values, y=med.index, color="steelblue")
    ax.bar_label(ax.containers[0], labels=[euro(v, 0) for v in med.values],
                 padding=3, fontsize=9)
    ax.grid(False)
    ax.set_xticks([])
    plt.margins(x=0.15)
    plt.title(titel); plt.xlabel("Medianpreis (EUR)"); plt.ylabel("")
    plt.tight_layout()
    plt.savefig(IMG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()



# Streudiagramm: Preis gegen ein numerisches Merkmal (mit Trendlinie)
def plot_streudiagramm(spalte, titel, xlabel, fname, xlim=None):
    stichprobe = df_clean.sample(8000, random_state=42)      # Stichprobe für Lesbarkeit
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=stichprobe, x=spalte, y="price_in_euro",
                    alpha=0.25, s=12, color="steelblue")
    sns.regplot(data=stichprobe, x=spalte, y="price_in_euro",
                scatter=False, color="red", line_kws={"linewidth": 1.5})
    if xlim:
        plt.xlim(*xlim)
    plt.ylim(0, 100000)                                      # Preis-Ausreißer für die Ansicht begrenzen
    plt.yticks(range(0, 100001, 10000))                      # Preis-Achse in 10.000er-Schritten
    plt.title(titel); plt.xlabel(xlabel); plt.ylabel("Preis (EUR)")
    plt.tight_layout()
    plt.savefig(IMG_DIR / fname, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Daten laden
df = pd.read_csv(DATA_RAW)
print("Shape:", df.shape)
df.head()

## 3. Datenqualität

### 3.1 Struktur & Datentypen

In [ ]:
# Überblick: Spalten, Datentypen (Dtype), vorhandene Werte
df.info()

`df.info()` zeigt, dass 13 der 15 Spalten als Text (`object`) gespeichert sind,
obwohl mehrere davon Zahlen enthalten. `year`, `power_kw` und `power_ps` sollten
Ganzzahlen (int) sein, `price_in_euro` und `fuel_consumption_l_100km` Kommazahlen
(float) und `registration_date` ein Datum. Die Spalte `Unnamed: 0` ist lediglich
ein überflüssiger Index. Die falschen Datentypen deuten auf verunreinigte Zellen
in einzelnen Zeilen hin; die Umwandlung erfolgt im späteren
Bereinigungsschritt. Zudem ist der Non-Null-Count nicht überall gleich: Mehrere Spalten enthalten
fehlende Werte, am deutlichsten `fuel_consumption_l_100km` mit nur 224.206 von
251.079 Einträgen. Diese werden in Kapitel 3.2 genauer untersucht.

### 3.2 Fehlende Werte

In [ ]:
# Anteil echter fehlender Werte (NaN) je Spalte in %, absteigend sortiert
(df.isna().mean() * 100).round(2).sort_values(ascending=False)

In [ ]:
# Versteckte Missings prüfen – Platzhalter statt NaN
df["fuel_consumption_g_km"].value_counts().head()

In [ ]:
df["transmission_type"].value_counts()

In [ ]:
# "0 g/km" plausibel nur bei Elektrofahrzeugen
mask = df["fuel_consumption_g_km"].str.strip() == "0 g/km"
df.loc[mask, "fuel_type"].value_counts()

In [ ]:
# "0 g/km" ist nur bei Nullemissions-Antrieben plausibel (Elektro, Wasserstoff)
mask_0 = df["fuel_consumption_g_km"].str.strip() == "0 g/km"
nullemission = ["Electric", "Hydrogen"]
unplausibel = (mask_0 & ~df["fuel_type"].isin(nullemission)).sum()
print("unplausibel 0 g/km (Verbrenner):", unplausibel)

Nur `fuel_consumption_l_100km` weist mit 10,7 % viele **echte** fehlende Werte (NaN) auf.
Zusätzlich existieren **versteckte** fehlende Werte, die `isna()` nicht erkennt.
`fuel_consumption_g_km` enthält 35.809-mal den Platzhalter `"- (g/km)"`.
In `transmission_type` steht 1.144-mal `"Unknown"`.
Die tatsächliche Fehlerquote ist damit höher, als der reine NaN-Check vermuten lässt.
Darüber hinaus enthält `fuel_consumption_g_km` 8.533-mal den Wert `0 g/km`.
Der Kreuzvergleich mit `fuel_type` zeigt, dass dies nur bei Elektrofahrzeugen
(2.930) und Wasserstofffahrzeugen (57) plausibel ist, während er bei Verbrennern (5.546) fehlerhaft ist.

Die Platzhalter werden im Bereinigungsschritt in echte NaN
umgewandelt. Aufgrund der hohen Fehlerquote und der Redundanz beider
Verbrauchsspalten werden diese als Prädiktoren kritisch geprüft.

### 3.3 Duplikate

In [ ]:
# Check mit künstlichen Index - irreführend, macht jede Zeile einzigartig
df.duplicated().sum()

In [ ]:
# Echte Duplikate: Anzahl und Anteil
dupes = df.drop(columns="Unnamed: 0").duplicated().sum()
print(f"Duplikate: {dupes} ({dupes / len(df) * 100:.2f} %)")

Der naive Duplikat-Check liefert 0, weil die künstliche Index-Spalte
`Unnamed: 0` jede Zeile eindeutig macht und echte Duplikate verdeckt.
Schließt man diese Spalte aus, zeigen sich 6.353 doppelte Inserate (2.53 %).

Die Duplikate werden im Bereinigungsschritt entfernt.

## 4. Datenbereinigung

### 4.1 Kopie (df_clean)  erstellen, Index & Duplikate entfernen

In [ ]:
# Rohdatenkopie, (df) bleiben unverändert
df_clean = df.copy()

# Überflüssige Index-Spalte entfernen
df_clean = df_clean.drop(columns="Unnamed: 0")

# Echte Duplikate entfernen
vorher = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Entfernte Duplikate: {vorher - len(df_clean)} | Verbleibend: {len(df_clean)}")

Als Erstes wird eine Kopie erstellt, um die Rohdaten unverändert zu
lassen. Die überflüssige Index-Spalte `Unnamed: 0` wird entfernt und die
6.353 echten Duplikate gelöscht und es verbleiben 244.726 Inserate.

### 4.2 Datentypen korrigieren

In [ ]:
# Preis als Zahl (float); errors="coerce" -> ungültiger Text wird NaN
df_clean["price_in_euro"] = pd.to_numeric(df_clean["price_in_euro"], errors="coerce")

# Ganzzahl-Merkmale als Int64 (Ganzzahl, verträgt aber NaN)
for col in ["year", "power_kw", "power_ps"]:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

In [ ]:
# Zulassungsdatum (Format MM/JJJJ) in echtes Datum umwandeln
df_clean["registration_date"] = pd.to_datetime(
    df_clean["registration_date"], format="%m/%Y", errors="coerce"
)

In [ ]:
# Prüfen: Typen korrekt? Wie viele NaN sind durch die Umwandlung entstanden?
print(df_clean.dtypes)
print("\nNeue NaN je Spalte:")
print(df_clean.isna().sum())

Nach der Umwandlung liegen alle Spalten im korrekten Typ vor (`year`, `power_kw`,
`power_ps` als Int64, `price_in_euro`/`mileage_in_km` als float und `registration_date`
als Datum). Durch `errors="coerce"` wurden die verschobenen Zeilen als NaN
markiert (z. B. 180 in `year`, 277 in `power_kw`). Diese werden in 4.3 behandelt.

### 4.3 Platzhalter & unplausible Werte behandeln

In [ ]:
# Versteckte Platzhalter in echte NaN umwandeln
df_clean["transmission_type"] = df_clean["transmission_type"].replace("Unknown", np.nan)
df_clean["fuel_type"] = df_clean["fuel_type"].replace("Unknown", np.nan)

In [ ]:
# Wertebereich von year prüfen
print(df_clean["year"].min(), df_clean["year"].max())

In [ ]:
# Wie viele Fahrzeuge haben ein unplausibles Baujahr (> 2023)?
print("Unplausible Jahre (>2023):", (df_clean["year"] > 2023).sum())

# Unplausible Jahre auf NaN setzen (Datensatz deckt nur 1995–2023 ab)
df_clean.loc[df_clean["year"] > 2023, "year"] = pd.NA

Die Platzhalter `"Unknown"` in `transmission_type` und `fuel_type` wurden in
echte NaN umgewandelt. Zudem wiesen 4 Fahrzeuge ein unplausibles Baujahr über
2023 auf (Datensatz deckt 1995–2023 ab); diese wurden auf NaN gesetzt.

### 4.4 Finale Spaltenauswahl & Export

In [ ]:
# Zwischenstand nach Typkorrektur
df_clean.head()

In [ ]:
# Prüfen: Ist das Jahr aus registration_date redundant zu year?
maske = df_clean["registration_date"].notna() & df_clean["year"].notna()
abweichungen = (df_clean.loc[maske, "registration_date"].dt.year
                != df_clean.loc[maske, "year"]).sum()
print("Verglichene Zeilen:", maske.sum(), "| Abweichende Jahre:", abweichungen)

In [ ]:
# Monat der Erstzulassung als eigenes numerisches Merkmal
df_clean["registration_month"] = df_clean["registration_date"].dt.month

# Kontrolle: Jahr + Monat sauber getrennt?
df_clean[["registration_date", "year", "registration_month"]].head()

In [ ]:
# Nicht benötigte Spalten entfernen
df_clean = df_clean.drop(columns=[
    "registration_date",         # atomar zerlegt in year + registration_month
    "offer_description",         # Freitext, kein Prädiktor
    "fuel_consumption_l_100km",  # ~11% fehlend
    "fuel_consumption_g_km",     # redundant + fehlerhafte Werte
    "power_kw",                  # redundant zu power_ps
])

# Zeilen mit fehlenden Werten entfernen
vorher = len(df_clean)
df_clean = df_clean.dropna()
print(f"Entfernte Zeilen: {vorher - len(df_clean)} | Verbleibend: {len(df_clean)}")

# Ganzzahl-Spalten final auf int (jetzt ohne NaN)
for col in ["year", "registration_month", "power_ps"]:
    df_clean[col] = df_clean[col].astype(int)

# Export
df_clean.to_csv(DATA_PROCESSED, index=False)
print("Gespeichert:", DATA_PROCESSED, "| Shape:", df_clean.shape)

In [ ]:
# Test: gespeicherte Datei zurücklesen und prüfen
df_check = pd.read_csv(DATA_PROCESSED)
print("Shape:", df_check.shape)
df_check.info()

In der finalen Auswahl wurden Spalten entfernt, die entweder redundant sind
(`power_kw` doppelt zu `power_ps`, `registration_date` zu `year`) oder stark
unvollständig (die beiden Verbrauchsspalten). Aus der Erstzulassung wurde der
Monat als eigenes numerisches Merkmal extrahiert, sodass Jahr und Monat atomar
getrennt vorliegen. Übrig bleiben 243.060 Inserate mit 10 aussagekräftigen
Merkmalen ohne fehlende Werte. Der Datensatz wurde als `data_processed.csv`
gespeichert; ein erneutes Einlesen bestätigte den fehlerfreien Export als
Grundlage für EDA und Modellierung.

## 5. Explorative Datenanalyse

### 5.1 Zielvariable: Preisverteilung

In [ ]:
# Statistische Kennzahlen der Zielvariable
desc = df_clean["price_in_euro"].describe().apply(euro)
desc["count"] = f"{int(df_clean['price_in_euro'].count()):,}".replace(",", ".")
desc

Die Kennzahlen zeigen eine deutliche Rechtsschiefe: Der Mittelwert (26.035 €)
liegt klar über dem Median (19.680 €), da wenige sehr teure Fahrzeuge den
Durchschnitt nach oben ziehen. Die mittleren 50 % der Fahrzeuge kosten zwischen
11.900 € und 29.900 €. Die große Spanne (120 € bis 5,89 Mio €) und die hohe
Standardabweichung (37.119 €) belegen die starke Streuung.

In [ ]:
plt.figure(figsize=(12, 4))
ax = sns.histplot(df_clean["price_in_euro"], bins=100, log_scale=True, color="steelblue")

ticks = [1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000]
ax.set_xticks(ticks)
ax.set_xlim(1000, 500000)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: euro(x, 0)))

plt.title("Verteilung der Fahrzeugpreise (logarithmische Preisachse)")
plt.xlabel("Preis (EUR)")
plt.ylabel("Anzahl")
plt.tight_layout()
plt.savefig(IMG_DIR / "01_preisverteilung.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Ausgeblendete Randbereiche: Anzahl und Anteil
unter = df_clean["price_in_euro"] < 1000
ueber = df_clean["price_in_euro"] > 500000
print(f"< 1.000 €:   {unter.sum()} Fahrzeuge ({unter.mean()*100:.2f} %)")
print(f"> 500.000 €: {ueber.sum()} Fahrzeuge ({ueber.mean()*100:.2f} %)")

Auf der logarithmischen Preisachse zeigt sich eine annähernd log-normale
Verteilung mit einem klaren Schwerpunkt um 20.000 €. Für die Darstellung sind
die Randbereiche unter 1.000 € (848 Fahrzeuge, 0,35 %) und über 500.000 €
(124 Fahrzeuge, 0,05 %) ausgeblendet; dabei handelt es sich um seltene, aber
reale Fahrzeuge.

### 5.2 Numerische Merkmale

In [ ]:
# Kennzahlen der numerischen Merkmale
df_clean[["mileage_in_km", "power_ps", "year"]].describe().round(0)

Die Kennzahlen zeigen deutliche Unterschiede zwischen den Merkmalen. Der
Kilometerstand ist stark rechtsschief (Median 69.100 km < Mittelwert 86.811 km)
und reicht bis unplausible 3,8 Mio km – ein klarer Ausreißer. Die Leistung liegt
im Median bei 150 PS; das Maximum von 999 PS wirkt als runder Wert verdächtig.
Das Baujahr konzentriert sich mit Median 2018 und geringer Streuung (SD 5 Jahre)
auf jüngere Fahrzeuge (1995–2023).

In [ ]:
plot_verteilung(df_clean["mileage_in_km"], "Kilometerstand", "km",
                "02_kilometerstand.png", bins=60, binrange=(0, 350000), xlim=(0, 350000))

plot_verteilung(df_clean["year"], "Baujahr", "Jahr",
                "04_baujahr.png", discrete=True,
                xlim=(1995, 2024),                         
                xticks=[1995, 2000, 2005, 2010, 2015, 2020, 2023])

plot_verteilung(df_clean["power_ps"], "Leistung", "PS",
                "03_leistung.png", bins=40, binrange=(0, 600), xlim=(50, 600),
                xticks=list(range(50, 601, 50)))      

# Anteil der in den Grafiken ausgeblendeten Randbereiche
ausgeblendet = {
    "Kilometerstand > 350.000 km": df_clean["mileage_in_km"] > 350000,
    "Leistung < 50 PS":            df_clean["power_ps"] < 50,
    "Leistung > 600 PS":           df_clean["power_ps"] > 600,
}

for bezeichnung, maske in ausgeblendet.items():
    print(f"{bezeichnung:<28}: {maske.sum():>5} Fahrzeuge ({maske.mean()*100:.2f} %)")

Kilometerstand und Leistung sind rechtsschief.  Die meisten Fahrzeuge haben unter
150.000 km bzw. 100–200 PS, mit auslaufenden Rändern nach oben. Der hohe Balken
bei niedrigem Kilometerstand deutet auf Neu- und Vorführwagen hin. Das Baujahr
ist dagegen linksschief – der Bestand besteht überwiegend aus neueren Fahrzeugen
mit Schwerpunkt ab etwa 2015.

Für die Darstellung wurden seltene Randbereiche ausgeblendet, nämlich ein
Kilometerstand über 350.000 km (0,53 %) sowie eine Leistung unter 50 PS (0,08 %)
und über 600 PS (0,72 %). Dabei handelt es sich um reale, aber seltene Fahrzeuge
und nicht um Datenfehler.

### 5.3 Kategoriale Merkmale

In [ ]:
for col in ["brand", "fuel_type", "transmission_type"]:
    print(f"{col}: {df_clean[col].nunique()} Ausprägungen")

In [ ]:
# Häufigkeiten
plot_kategorie("brand", "Häufigste Marken (Top 15)", "05_marken.png", top=15, capitalize=True)
plot_kategorie("fuel_type", "Kraftstoffart", "06_kraftstoff.png")
plot_kategorie("transmission_type", "Getriebe", "07_getriebe.png")

Der Fahrzeugbestand wird klar von deutschen Marken dominiert: Volkswagen,
Mercedes-Benz und Audi stellen mit Abstand die größten Anteile, gefolgt von Opel,
BMW und Ford. Bei der Kraftstoffart überwiegen Benziner und Diesel deutlich,
während alternative Antriebe wie Hybrid und Elektro noch geringe Anteile
ausmachen; sehr seltene Antriebe wie Wasserstoff oder Ethanol treten kaum auf.
Beim Getriebe halten sich Automatik und Schaltgetriebe etwa die Waage, wohingegen
die Semi-Automatik nahezu keine Rolle spielt.

In [ ]:
# Preisbezug
plot_preis_je_kategorie("brand", "Medianpreis der häufigsten Marken", "10_preis_marke.png", top_haeufig=15, capitalize=True)
plot_preis_je_kategorie("fuel_type", "Medianpreis nach Kraftstoffart", "08_preis_kraftstoff.png")
plot_preis_je_kategorie("transmission_type", "Medianpreis nach Getriebe", "09_preis_getriebe.png")

Die drei Grafiken zeigen, welche Kategorien mit höheren Preisen einhergehen.
Bei den Marken liegen Toyota, Audi und Mercedes-Benz preislich vorn, Opel, Fiat
und Volkswagen dagegen hinten. Interessant ist Volkswagen ist die häufigste Marke im
Datensatz, preislich aber nur im unteren Drittel. VW verkauft vor allem günstige
Modelle für den Massenmarkt.

Bei der Kraftstoffart sind Elektro- und Hybridfahrzeuge im Median teurer als
Benziner und Diesel. Die noch höheren Werte einzelner Antriebe (Diesel-Hybrid,
Wasserstoff, Ethanol) sind wenig aussagekräftig, da sie auf nur sehr wenigen
Fahrzeugen beruhen.

Am deutlichsten ist der Effekt beim Getriebe: Fahrzeuge mit Automatik kosten im
Median fast doppelt so viel wie Fahrzeuge mit Schaltgetriebe (26.990 € gegenüber
13.750 €).

In [ ]:
farben_de = {"black": "Schwarz", "grey": "Grau", "white": "Weiß", "silver": "Silber",
             "blue": "Blau", "red": "Rot", "brown": "Braun", "green": "Grün",
             "orange": "Orange", "beige": "Beige", "yellow": "Gelb", "gold": "Gold",
             "bronze": "Bronze", "violet": "Violett"}

plot_kategorie("color", "Fahrzeugfarbe nach Häufigkeit", "15_farbe.png", mapping=farben_de)
plot_preis_je_kategorie("color", "Medianpreis nach Farbe", "16_preis_farbe.png", mapping=farben_de)

Der Preisunterschied zwischen den Farben ist vergleichsweise gering. Zwar erzielen
Gold, Grün und Bronze die höchsten Medianpreise, doch diese Farben sind selten,
sodass ihre Werte auf wenigen Fahrzeugen beruhen und wenig aussagekräftig sind. Bei
den häufigen Farben liegen die Medianpreise dicht beieinander. Die Farbe ist damit
ein schwacher Preistreiber. Die Unterschiede spiegeln vermutlich eher wider, welche
Fahrzeugtypen in welchen Farben angeboten werden, als einen direkten Effekt der
Farbe selbst.

### 5.4 Zusammenhänge & Korrelationen

In [ ]:
# Korrelationsmatrix mit lesbaren deutschen Bezeichnungen
labels = {"price_in_euro": "Preis", "mileage_in_km": "Kilometerstand",
          "power_ps": "Leistung", "year": "Baujahr", "registration_month": "Monat"}
korr = df_clean[list(labels.keys())].corr()
korr.index = korr.columns = list(labels.values())

plt.figure(figsize=(8, 6))
sns.heatmap(korr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            annot_kws={"size": 13}, linewidths=0.5, square=True)
plt.yticks(rotation=0)
plt.title("Korrelationsmatrix der numerischen Merkmale")
plt.tight_layout()
plt.savefig(IMG_DIR / "11_korrelation.png", dpi=150, bbox_inches="tight")
plt.show()

Die Korrelationsmatrix zeigt die linearen Zusammenhänge zwischen den numerischen
Merkmalen. Am stärksten hängt die Leistung mit dem Preis zusammen, mit einem Wert
von 0,62. Der Kilometerstand wirkt negativ auf den Preis und das Baujahr positiv,
beide mit moderater Stärke. Der Zulassungsmonat hat praktisch keinen Einfluss.
Auffällig ist die starke negative Korrelation zwischen Kilometerstand und Baujahr
mit einem Wert von 0,71. Neuere Fahrzeuge haben also weniger Kilometer, sodass beide
Merkmale teilweise dasselbe messen und in der Regression zu Multikollinearität
führen können.

In [ ]:
plot_streudiagramm("mileage_in_km", "Preis vs. Kilometerstand", "km", "12_scatter_km.png", xlim=(0, 300000))
plot_streudiagramm("power_ps", "Preis vs. Leistung", "PS", "13_scatter_ps.png", xlim=(50, 400))
     # Preis-Achse in 10.000er-Schritten

Der Kilometerstand zeigt einen klaren negativen Zusammenhang mit dem Preis. Je
höher die Laufleistung ist, desto günstiger ist das Fahrzeug.

Bei der Leistung
verhält es sich umgekehrt. Mehr PS gehen mit höheren Preisen einher.

Beide
Zusammenhänge sind plausibel und entsprechen der Erwartung am Gebrauchtwagenmarkt.
Die breite Streuung zeigt zugleich, dass der Preis von weiteren Faktoren abhängt.

## 6. Zwischenfazit

Die explorative Analyse zeigt, dass der Gebrauchtwagenpreis von mehreren Faktoren
gemeinsam bestimmt wird. Unter den numerischen Merkmalen hängt die Leistung am
stärksten mit dem Preis zusammen, gefolgt von Kilometerstand und Baujahr. Der
Zulassungsmonat spielt dagegen keine erkennbare Rolle. Bei den kategorialen
Merkmalen zeigt sich der deutlichste Unterschied beim Getriebe, da Fahrzeuge mit
Automatik im Median fast doppelt so teuer sind wie solche mit Schaltgetriebe. Auch
die Marke wirkt sich klar aus, wobei Häufigkeit und Preisniveau nicht
zusammenfallen. Volkswagen ist die häufigste Marke, liegt preislich aber nur im
unteren Bereich. Die Farbe erweist sich als schwacher Preistreiber, dessen
Unterschiede eher den angebotenen Fahrzeugtypen als der Farbe selbst zuzuschreiben
sind.

Die Preisverteilung ist stark rechtsschief und annähernd log-normal. Für Vergleiche
wurde deshalb der robuste Median verwendet, und für die Regression ist eine
logarithmische Transformation der Zielvariable zu erwägen. Die breite Streuung in
den Streudiagrammen bestätigt, dass kein einzelnes Merkmal den Preis allein erklärt.

Bei der Datenaufbereitung mussten mehrere Qualitätsprobleme behandelt werden. Dazu
gehörten verschobene Zeilen aus dem Scraping, versteckte Platzhalter und unplausible
Werte. Extreme Preise wurden nicht gelöscht, da es sich um realeLuxusfahrzeuge 
handelt, sondern nur für die Darstellung ausgeblendet. Eine wichtige Beobachtung
für die Modellierung ist die starke Korrelation zwischen Kilometerstand und Baujahr,
da beide Merkmale teilweise dasselbe messen und zu Multikollinearität führen können.

Insgesamt liefert die explorative Analyse eine belastbare Grundlage für die
anschließende Modellierung. Die Ergebnisse legen nahe, dass sich der Preis vor allem
mit Leistung, Kilometerstand, Baujahr, Marke und Getriebe erklären lässt, dass dafür
aber ein Modell mit mehreren Merkmalen nötig ist. Merkmale ohne erkennbaren Einfluss
wie der Zulassungsmonat sowie das Merkmal model mit seiner sehr hohen Anzahl an
Ausprägungen werden in der Modellierung gesondert betrachtet.